# Bases analíticas da Fase 3

Este notebook cria dois arquivos em `data/processed`:

- `base_analitica_municipio.parquet`: uma linha por ano-alvo, município e rede.
- `base_analitica_aluno.parquet`: uma linha por aluno presente com prova preenchida.

A montagem respeita o alinhamento temporal. Para o ano-alvo `t`, os indicadores de alfabetização e o IDEB entram somente com edições anteriores a `t`. As colunas contemporâneas da avaliação, como proficiência individual e agregados da mesma prova, não entram como features.

Execute as células na ordem.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd


## 1. Diretórios e configuração


In [2]:
def localizar_raiz_projeto(inicio: Path | None = None) -> Path:
    atual = (inicio or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "data" / "gold").exists():
            return candidato
    raise FileNotFoundError(
        "Não foi encontrada a pasta data/gold. Abra o notebook dentro do projeto "
        "ou defina PROJECT_ROOT manualmente."
    )

PROJECT_ROOT = localizar_raiz_projeto()
GOLD_DIR = PROJECT_ROOT / "data" / "gold"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ARQUIVO_MUNICIPIO = PROCESSED_DIR / "base_analitica_municipio.parquet"
ARQUIVO_ALUNO = PROCESSED_DIR / "base_analitica_aluno.parquet"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Gold: {GOLD_DIR}")
print(f"Processed: {PROCESSED_DIR}")


Raiz do projeto: /home/linux/projetos/Teste_02112026
Gold: /home/linux/projetos/Teste_02112026/data/gold
Processed: /home/linux/projetos/Teste_02112026/data/processed


## 2. Funções auxiliares


In [3]:
MAPA_REDE_CODIGO = {
    1: "federal",
    2: "estadual",
    3: "municipal",
    4: "privada",
}

COLUNAS_ALVO_MUNICIPIO = ["indicador_t", "atingiu_meta_t", "gap_meta_t"]


def ler_gold(nome: str) -> pd.DataFrame:
    caminho = GOLD_DIR / nome
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo Gold não encontrado: {caminho}")
    return pd.read_parquet(caminho)


def normalizar_rede(serie: pd.Series) -> pd.Series:
    return (
        serie.astype("string")
        .str.strip()
        .str.lower()
        .replace({"pública": "publica", "público": "publica"})
    )


def validar_unicidade(df: pd.DataFrame, chave: list[str], nome: str) -> None:
    nulos = df[chave].isna().any(axis=1)
    duplicados = df.duplicated(chave, keep=False)
    if nulos.any():
        raise ValueError(f"{nome}: {int(nulos.sum())} linhas com chave nula em {chave}")
    if duplicados.any():
        exemplos = df.loc[duplicados, chave].head().to_dict("records")
        raise ValueError(f"{nome}: chave duplicada em {chave}; exemplos={exemplos}")


def ultimo_historico_antes(
    historico: pd.DataFrame,
    ano_alvo: int,
    chaves_grupo: list[str],
    prefixo: str,
    colunas_valor: list[str],
) -> pd.DataFrame:
    hist = historico[historico["ano"] <= ano_alvo - 1].copy()
    hist = hist.sort_values(chaves_grupo + ["ano"])
    ultimo = hist.groupby(chaves_grupo, dropna=False).tail(1).copy()
    anterior = hist.groupby(chaves_grupo, dropna=False).nth(-2).reset_index()

    ren_ultimo = {c: f"{prefixo}_{c}_ultimo" for c in colunas_valor}
    ultimo = ultimo[chaves_grupo + ["ano"] + colunas_valor].rename(
        columns={"ano": f"ref_{prefixo}", **ren_ultimo}
    )

    if colunas_valor:
        principal = colunas_valor[0]
        anterior = anterior[chaves_grupo + [principal]].rename(
            columns={principal: f"{prefixo}_{principal}_anterior"}
        )
        ultimo = ultimo.merge(anterior, on=chaves_grupo, how="left")
        ultimo[f"{prefixo}_{principal}_variacao"] = (
            ultimo[f"{prefixo}_{principal}_ultimo"]
            - ultimo[f"{prefixo}_{principal}_anterior"]
        )
    return ultimo


## 3. Base analítica municipal


In [4]:
def bloco_indicador_municipal(
    indicador: pd.DataFrame,
    metas: pd.DataFrame,
    ano_alvo: int,
) -> pd.DataFrame:
    chave = ["id_municipio", "rede"]

    t1 = indicador[indicador["ano"] == ano_alvo - 1].set_index(chave)
    t2 = indicador[indicador["ano"] == ano_alvo - 2].set_index(chave)
    alvo = indicador[indicador["ano"] == ano_alvo].set_index(chave)
    meta_t = metas[metas["ano"] == ano_alvo].set_index(chave)
    meta_t1 = metas[metas["ano"] == ano_alvo - 1].set_index(chave)

    indices = t1.index.union(meta_t.index).union(alvo.index)
    out = pd.DataFrame(index=indices)
    out["indicador_t1"] = t1["indicador_pct"]
    out["nivel_t1"] = t1["nivel"]
    out["participacao_t1"] = t1["participacao_pct"]
    out["indicador_t2"] = t2["indicador_pct"]
    out["variacao_t1"] = out["indicador_t1"] - out["indicador_t2"]
    out["meta_t1"] = meta_t1["meta_pct"]
    out["gap_meta_t1"] = out["indicador_t1"] - out["meta_t1"]
    out["meta_t"] = meta_t["meta_pct"]
    out["salto_necessario"] = out["meta_t"] - out["indicador_t1"]

    out["indicador_t"] = alvo["indicador_pct"]
    out["atingiu_meta_t"] = alvo["atingiu_meta"]
    out["gap_meta_t"] = alvo["gap_meta_pp"]
    return out.reset_index()


def build_base_analitica_municipio() -> pd.DataFrame:
    dim = ler_gold("dim_municipio.parquet").drop(columns="_gold_processed_at", errors="ignore")
    contexto = ler_gold("contexto_municipio.parquet").drop(columns="_gold_processed_at", errors="ignore")
    ideb = ler_gold("ideb_municipio.parquet").drop(columns="_gold_processed_at", errors="ignore")
    indicador = ler_gold("indicador_municipio.parquet").drop(columns="_gold_processed_at", errors="ignore")
    metas = ler_gold("metas_municipio.parquet").drop(columns="_gold_processed_at", errors="ignore")

    indicador["rede"] = normalizar_rede(indicador["rede"])
    metas["rede"] = normalizar_rede(metas["rede"])
    ideb["rede"] = normalizar_rede(ideb["rede"])

    anos_alvo = sorted(
        set(pd.to_numeric(indicador["ano"], errors="coerce").dropna().astype(int))
        | set(pd.to_numeric(metas["ano"], errors="coerce").dropna().astype(int))
    )

    partes = []
    valores_ideb = [
        "ideb_observado",
        "ideb_nota_portugues",
        "ideb_nota_matematica",
        "ideb_nota_media",
        "ideb_rendimento",
        "ideb_meta",
        "aprovacao_ai_pct",
        "aprovacao_1ano_pct",
        "aprovacao_2ano_pct",
    ]

    for ano_alvo in anos_alvo:
        base = bloco_indicador_municipal(indicador, metas, ano_alvo)
        base.insert(0, "ano_alvo", ano_alvo)

        ideb_hist = ultimo_historico_antes(
            ideb,
            ano_alvo,
            ["id_municipio", "rede"],
            "ideb",
            valores_ideb,
        )
        base = base.merge(
            ideb_hist,
            on=["id_municipio", "rede"],
            how="left",
            validate="one_to_one",
        )
        partes.append(base)

    abt = pd.concat(partes, ignore_index=True)
    abt = abt.merge(contexto, on="id_municipio", how="left", validate="many_to_one")

    # A dimensão já está contida no contexto. O merge abaixo só completa nomes ausentes.
    dim_cols = ["id_municipio", "nome_municipio", "sigla_uf", "id_uf", "nome_uf", "nome_regiao"]
    faltantes_dim = [c for c in dim_cols if c not in abt.columns]
    if faltantes_dim:
        abt = abt.merge(dim[dim_cols], on="id_municipio", how="left", validate="many_to_one")

    abt = abt.dropna(subset=["id_municipio", "sigla_uf"])
    colunas_frente = [
        "ano_alvo", "id_municipio", "rede", "nome_municipio",
        "sigla_uf", "id_uf", "nome_uf", "nome_regiao",
    ]
    colunas_frente = [c for c in colunas_frente if c in abt.columns]
    outras = [c for c in abt.columns if c not in colunas_frente + COLUNAS_ALVO_MUNICIPIO]
    abt = abt[colunas_frente + outras + COLUNAS_ALVO_MUNICIPIO]
    abt = abt.sort_values(["ano_alvo", "id_municipio", "rede"]).reset_index(drop=True)
    validar_unicidade(abt, ["ano_alvo", "id_municipio", "rede"], "base_analitica_municipio")
    return abt


## 4. Base analítica de alunos


In [5]:
FEATURES_MUNICIPAIS_EXCLUIDAS_DO_ALUNO = [
    "indicador_t",
    "atingiu_meta_t",
    "gap_meta_t",
    "nome_municipio",
    "nome_uf",
    "sigla_uf",
    "id_uf",
    "nome_regiao",
]


def build_base_analitica_aluno(
    base_municipio: pd.DataFrame,
) -> pd.DataFrame:
    alunos = ler_gold("fato_aluno.parquet").drop(columns="_gold_processed_at", errors="ignore")

    presentes = pd.to_numeric(alunos["presente"], errors="coerce").eq(1)
    preenchidos = pd.to_numeric(alunos["in_preenchimento_lp"], errors="coerce").eq(1)
    p = alunos.loc[presentes & preenchidos].copy()
    p = p.dropna(subset=["ano", "id_aluno", "id_municipio", "id_escola", "alfabetizado"])

    p["rede_codigo"] = pd.to_numeric(p["rede_codigo"], errors="coerce").astype("Int64")
    p["rede"] = p["rede_codigo"].map(MAPA_REDE_CODIGO).astype("string")

    # Features da escola calculadas sem usar proficiência ou rótulo agregado.
    todos = alunos.dropna(subset=["ano", "id_escola"]).copy()
    todos["presente_num"] = pd.to_numeric(todos["presente"], errors="coerce")
    escola = todos.groupby(["ano", "id_escola"], dropna=False).agg(
        escola_qtd_registros=("id_aluno", "size"),
        escola_taxa_presenca=("presente_num", "mean"),
    ).reset_index()
    escola["escola_taxa_presenca"] = 100 * escola["escola_taxa_presenca"]

    avaliados = p.groupby(["ano", "id_escola"], dropna=False).size().rename(
        "escola_qtd_avaliados"
    ).reset_index()
    escola = escola.merge(avaliados, on=["ano", "id_escola"], how="left")
    p = p.merge(escola, on=["ano", "id_escola"], how="left", validate="many_to_one")

    # Remove variáveis contemporâneas que revelam diretamente o desempenho da prova.
    p = p.drop(
        columns=[
            "proficiencia",
            "co_caderno_lp",
            "vl_peso_aluno_lp",
            "nu_ano_avaliacao",
            "presente",
            "in_preenchimento_lp",
        ],
        errors="ignore",
    )

    municipio = base_municipio.drop(
        columns=FEATURES_MUNICIPAIS_EXCLUIDAS_DO_ALUNO,
        errors="ignore",
    ).rename(columns={"ano_alvo": "ano"})

    # Join por rede quando o código da dependência permite identificar a rede.
    p = p.merge(
        municipio,
        on=["ano", "id_municipio", "rede"],
        how="left",
        validate="many_to_one",
        suffixes=("", "_municipio"),
    )

    colunas_frente = [
        "ano", "id_aluno", "id_municipio", "id_escola",
        "sigla_uf", "rede_codigo", "rede", "tp_serie",
    ]
    colunas_frente = [c for c in colunas_frente if c in p.columns]
    target = ["alfabetizado"]
    outras = [c for c in p.columns if c not in colunas_frente + target]
    p = p[colunas_frente + outras + target]
    p = p.sort_values(["ano", "id_municipio", "id_escola", "id_aluno"]).reset_index(drop=True)
    validar_unicidade(p, ["ano", "id_aluno"], "base_analitica_aluno")
    return p


## 5. Criar e gravar os dois arquivos


In [6]:
base_analitica_municipio = build_base_analitica_municipio()
base_analitica_municipio.to_parquet(
    ARQUIVO_MUNICIPIO,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

base_analitica_aluno = build_base_analitica_aluno(base_analitica_municipio)
base_analitica_aluno.to_parquet(
    ARQUIVO_ALUNO,
    index=False,
    engine="pyarrow",
    compression="zstd",
)

print(
    f"[processed] base_analitica_municipio.parquet: "
    f"{len(base_analitica_municipio):,} linhas x "
    f"{base_analitica_municipio.shape[1]} colunas"
)
print(
    f"[processed] base_analitica_aluno.parquet: "
    f"{len(base_analitica_aluno):,} linhas x "
    f"{base_analitica_aluno.shape[1]} colunas"
)
print(f"Arquivos gravados em: {PROCESSED_DIR}")


[processed] base_analitica_municipio.parquet: 43,788 linhas x 59 colunas
[processed] base_analitica_aluno.parquet: 5,320,756 linhas x 61 colunas
Arquivos gravados em: /home/linux/projetos/Teste_02112026/data/processed


## 6. Validação final


In [7]:
resumo = pd.DataFrame([
    {
        "arquivo": ARQUIVO_MUNICIPIO.name,
        "linhas": len(base_analitica_municipio),
        "colunas": base_analitica_municipio.shape[1],
        "duplicidades_chave": int(
            base_analitica_municipio.duplicated(
                ["ano_alvo", "id_municipio", "rede"]
            ).sum()
        ),
        "target_nulo_pct": round(
            100 * base_analitica_municipio["atingiu_meta_t"].isna().mean(), 2
        ),
    },
    {
        "arquivo": ARQUIVO_ALUNO.name,
        "linhas": len(base_analitica_aluno),
        "colunas": base_analitica_aluno.shape[1],
        "duplicidades_chave": int(
            base_analitica_aluno.duplicated(["ano", "id_aluno"]).sum()
        ),
        "target_nulo_pct": round(
            100 * base_analitica_aluno["alfabetizado"].isna().mean(), 2
        ),
    },
])
display(resumo)

display(base_analitica_municipio.head())
display(base_analitica_aluno.head())


,arquivo,linhas,colunas,duplicidades_chave,target_nulo_pct
0,base_analitica_municipio.parquet,43788,59,0,75.37
1,base_analitica_aluno.parquet,5320756,61,0,0.00


,ano_alvo,id_municipio,rede,nome_municipio,sigla_uf,id_uf,nome_uf,nome_regiao,indicador_t1,nivel_t1,...,pc_nivel_8,pib_per_capita_reais,log_populacao,participacao_vab_agro_pct,participacao_vab_industria_pct,participacao_vab_servicos_pct,participacao_vab_adm_publica_pct,indicador_t,atingiu_meta_t,gap_meta_t
0,2023,1100015,municipal,Alta Floresta D'Oeste,RO,11,Rondônia,Norte,<NA>,<NA>,...,NaN,34170.791849,9.975576,0.006192,0.000554,0.003442,0.003428,65.0,<NA>,<NA>
1,2023,1100023,municipal,Ariquemes,RO,11,Rondônia,Norte,<NA>,<NA>,...,0.97,33163.219150,11.480753,0.000327,0.000455,0.000873,0.001459,62.0,<NA>,<NA>
2,2023,1100031,municipal,Cabixi,RO,11,Rondônia,Norte,<NA>,<NA>,...,NaN,44555.036442,8.585226,0.025225,0.001497,0.008598,0.006619,69.0,<NA>,<NA>
3,2023,1100049,municipal,Cacoal,RO,11,Rondônia,Norte,<NA>,<NA>,...,0.58,32139.514542,11.372375,0.000507,0.000352,0.000936,0.001785,63.0,<NA>,<NA>
4,2023,1100056,municipal,Cerejeiras,RO,11,Rondônia,Norte,<NA>,<NA>,...,0.97,46762.869729,9.673508,0.003340,0.000604,0.002717,0.006798,59.0,<NA>,<NA>


,ano,id_aluno,id_municipio,id_escola,sigla_uf,rede_codigo,rede,tp_serie,co_uf,escola_qtd_registros,...,pc_nivel_6,pc_nivel_7,pc_nivel_8,pib_per_capita_reais,log_populacao,participacao_vab_agro_pct,participacao_vab_industria_pct,participacao_vab_servicos_pct,participacao_vab_adm_publica_pct,alfabetizado
0,2023,11019569,1100015,60000156,RO,3,municipal,2,11,16,...,4.26,NaN,NaN,34170.791849,9.975576,0.006192,0.000554,0.003442,0.003428,0
1,2023,11019572,1100015,60000156,RO,3,municipal,2,11,16,...,4.26,NaN,NaN,34170.791849,9.975576,0.006192,0.000554,0.003442,0.003428,0
2,2023,11019579,1100015,60000156,RO,3,municipal,2,11,16,...,4.26,NaN,NaN,34170.791849,9.975576,0.006192,0.000554,0.003442,0.003428,0
3,2023,11019584,1100015,60000156,RO,3,municipal,2,11,16,...,4.26,NaN,NaN,34170.791849,9.975576,0.006192,0.000554,0.003442,0.003428,0
4,2023,11019585,1100015,60000159,RO,3,municipal,2,11,26,...,4.26,NaN,NaN,34170.791849,9.975576,0.006192,0.000554,0.003442,0.003428,1
